In [18]:
import re
from collections import OrderedDict

def parse_plant_traits(file_path):
    """
    Parses the plant trait description file and returns a list of plant dictionaries.
    """
    with open(file_path, 'r') as f:
        content = f.read()

    # Split by the functional type header
    plant_blocks = re.split(r'={5,}', content)
    
    plants = []
    for block in plant_blocks:
        if "Plant name" not in block:
            continue
            
        data = OrderedDict()
        lines = block.strip().split('\n')
        
        # Extract Plant Name
        name_match = re.search(r'Plant name\s+:\s*(.*)', block)
        plant_name = name_match.group(1).strip() if name_match else "Unknown"
        
        # Pattern to capture Description : Value
        # Uses \\: to treat the colon literally in the regex
        param_pattern = re.compile(r'^(.*?)\s+:\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?(?:\s+[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)*)$')
        
        for line in lines:
            # To remove specific brackets: [ and ]
            clean_line = re.sub(r'[\[\]]', '', line)
            # Remove leading index markers like 1| or 1:
            clean_line = re.sub(r'^\s*\d+[:|]', '', clean_line).strip()
            
            match = param_pattern.match(clean_line)
            if match:
                desc = match.group(1).strip()
                value = match.group(2).strip()
                data[desc] = value
                
        plants.append({'name': plant_name, 'params': data})
    
    return plants

def compare_plants(plants):
    """
    Compares two plants and highlights every parameter where values differ.
    """
    if len(plants) < 2:
        print("Error: Need at least two plant types in the file to perform a comparison.")
        return

    p1, p2 = plants[0], plants[1]
    
    # Define column widths
    desc_w = 80
    val_w = 30
    
    # Header
    title = f" COMPARISON: {p1['name']} vs {p2['name']} "
    print(f"\n{title:=^{desc_w + (val_w * 2) + 10}}")
    print(f"{'Parameter Description':<{desc_w}} | {'Value (Plant 1)':<{val_w}} | {'Value (Plant 2)':<{val_w}}")
    print("-" * (desc_w + (val_w * 2) + 10))

    diff_count = 0
    all_keys = list(p1['params'].keys())
    # Add any keys that might exist in plant 2 but not plant 1
    for k in p2['params'].keys():
        if k not in all_keys:
            all_keys.append(k)

    for key in all_keys:
        val1 = p1['params'].get(key, "MISSING")
        val2 = p2['params'].get(key, "MISSING")
        
        # Standardize whitespace for comparison (handles multi-value fields)
        is_different = " ".join(val1.split()) != " ".join(val2.split())
        
        diff_marker = " [ DIFF ] " if is_different else ""
        if is_different:
            diff_count += 1
            # Optional: Bold the line in terminals that support ANSI
            print(f"\033[1m{key[:desc_w-1]:<{desc_w}} | {val1:<{val_w}} | {val2:<{val_w}} \033[0m")
        else:
            print(f"{key[:desc_w-1]:<{desc_w}} | {val1:<{val_w}} | {val2:<{val_w}} |")

    print("-" * (desc_w + (val_w * 2) + 10))
    print(f"Total Parameters Compared: {len(all_keys)}")
    print(f"Total Differences Found: {diff_count}")
    
if __name__ == "__main__":
    # Ensure the filename matches your uploaded file
    file_name = '/Users/jinyuntang/work/github/ecosim_workspace/benchmark/smallset/MeditteraneanPastureCA/plant_trait.1981.desc'
    
    try:
        plant_data = parse_plant_traits(file_name)
        compare_plants(plant_data)
    except FileNotFoundError:
        print(f"Error: {file_name} not found. Please ensure the file is in the same directory.")


================================================= COMPARISON: C3 grass annual vs clover annual crop ==================================================
Parameter Description                                                            | Value (Plant 1)                | Value (Plant 2)               
------------------------------------------------------------------------------------------------------------------------------------------------------
VCMX     |  Saturated specific carboxylation rate by Rubisco at 25oC  umol CO2   | 40.000000000000000             | 55.000000000000000             
VOMX     |  Saturated specific oxygenation rate  at 25oC by Rubisco  umol O2 (g  | 13.000000000000000             | 12.000000000000000             
XKCO2    |  Reference Km for rubisco carboxylation of CO2  at 25oC  uM           | 12.500000000000000             | 12.500000000000000             |
XKO2     |  Reference Km for rubisco oxygenation  at 25oC  uM                    | 500.00000000000000    